In [1]:
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU") # Отключение GPU для TensorFlow

import os
import jax
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.8'

from td_sa_stack import get_dataset, TrainerModuleSingle, RegressionInceptionNetV1

%load_ext autoreload
%autoreload 2

2025-06-14 15:49:08.284997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749916148.304943   43737 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749916148.311459   43737 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749916148.326546   43737 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749916148.326560   43737 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749916148.326562   43737 computation_placer.cc:177] computation placer alr

In [2]:
import sys
from absl import flags
from ml_collections.config_flags import config_flags

sys.argv = [
    "",
    "--config=td_sa_stack/config.py",
]

config_flags.DEFINE_config_file("config", None, "Training configuration.", lock_config=True)
# flags.DEFINE_string("workdir", None, "Work directory.")
# flags.DEFINE_enum("mode", None, ["train", "eval", "fid_stats"], "Running mode: train, eval or fid_stats")
# flags.DEFINE_string("eval_folder", "eval", "The folder name for storing evaluation results")


FLAGS = flags.FLAGS
FLAGS(sys.argv)

config = FLAGS.config

In [3]:
config.multi_device == jax.device_count() > 1
config.model

activation: swish
name: RegressionInceptionNetV1
optimizer: adamw
optimizer_weight_decay: 1.0e-05

In [4]:
jax.devices()

[CudaDevice(id=0)]

In [5]:
train_ds, _, _ = get_dataset(config)

trainer = TrainerModuleSingle(config=config,
                              model_class=RegressionInceptionNetV1,
                              version=2)

Batch dimensions: [128]


Initializing model with batch shape: (128, 32, 32, 6)


/workspace/.venv/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Loaded model from step 11000


In [ ]:
trainer.train_model(train_ds=train_ds)

2025-06-14 15:50:04.852754: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
